# Part 3 — Silver Transformations

**Silver** = clean, reliable data.

Bronze stores data "as ingested from CSV". Coming on in Silver:
- Read data from Bronze-level;
- Check missing values and flight statuses;;
- Remove **duplicate-records**;
- Save cleaned data for future ML and BI tasks.

`CANCELLED` and `DIVERTED` flights are kept for further analysis.

In [0]:
from pyspark.sql import functions as F
#========================================
SCHEMA = "flight_delay"
BRONZE_FLIGHTS = f"{SCHEMA}.flights"
BRONZE_AIRLINES = f"{SCHEMA}.airlines"
BRONZE_AIRPORTS = f"{SCHEMA}.airports"
SILVER_FLIGHTS = f"{SCHEMA}.silver_flights"
SILVER_AIRLINES = f"{SCHEMA}.silver_airlines"
SILVER_AIRPORTS = f"{SCHEMA}.silver_airports"

## 1. Read Bronze

Silver reads from the Bronze Delta table.

In [0]:
flights = spark.table(BRONZE_FLIGHTS)
bronze_count = flights.count()
print(f"Bronze flights: {bronze_count:,} rows")

## 2. Checking flight statuses and missing delays

`CANCELLED = 1` — the flight was **cancelled**, not delayed.

`DIVERTED = 1` — the flight landed at a **different airport**.

And also emphasizing `NULL`s in `DEPARTURE_DELAY`.

We count these records, but don't remove them, because they may be useful for BI.

In [0]:
cancelled_count = flights.filter(F.col("CANCELLED") == 1).count()
diverted_count = flights.filter(F.col("DIVERTED") == 1).count()
null_delay_count = flights.filter(F.col("DEPARTURE_DELAY").isNull()).count()

print(f"Cancelled:  {cancelled_count:,}")
print(f"Diverted:   {diverted_count:,}")
print(f"Null Delay: {null_delay_count:,}")

## 3. Remove duplicates

Removing with escaping of metadata values. Because if metadata is different, but values of entire columns are equal, this is still a duplicate.

In [0]:
before = flights.count()
data_columns = [
    col for col in flights.columns
    if not col.startswith("_")
]
flights = flights.dropDuplicates(data_columns)
after_dedup = flights.count()
print(f"After dropDuplicates: {after_dedup:,} rows")
print(f"Removed: {before - after_dedup:,}")

## 3. Data quality summary


Comparing records amount in Bronze and Silver to define, how many duplicates were removed.

In [0]:
print("=== Data quality summary ===")
print(f"Bronze rows:        {bronze_count:,}")
print(f"Silver rows:        {after_dedup:,}")
print(f"Duplicates removed: {bronze_count - after_dedup:,}")
print(f"Cancelled:          {cancelled_count:,}")
print(f"Diverted:           {diverted_count:,}")
print(f"Null Delay:         {null_delay_count:,}")
print(f"Retained:           {after_dedup / bronze_count:.1%}")

## 4. Save Silver flights

In [0]:
flights.write.format("delta").mode("overwrite").saveAsTable(SILVER_FLIGHTS)
print(f"Saved: {SILVER_FLIGHTS}")
spark.table(SILVER_FLIGHTS).count()

## 5. Verify Silver flights

In [0]:
silver_check = spark.table(SILVER_FLIGHTS)
saved_count = silver_check.count()
print(f"Rows in Silver: {silver_check.count():,}")
assert saved_count == after_dedup, (
    f"Row amount mismatch; estimated {after_dedup:,}, got {saved_count:,} though"
)
silver_check.select(
    "AIRLINE", "ORIGIN_AIRPORT", "DEPARTURE_DELAY", "CANCELLED", "DIVERTED").show(5)

## 6. Silver for lookup tables

Airlines and airports are already clean — just dropping duplicate rows now. This would be helpful for adding airline and airport info in furhter analysis.

In [0]:
airlines_silver = spark.table(BRONZE_AIRLINES).dropDuplicates()
airports_silver = spark.table(BRONZE_AIRPORTS).dropDuplicates()
airlines_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_AIRLINES)
airports_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_AIRPORTS)
print(f"Saved: {SILVER_AIRLINES} ({airlines_silver.count()} rows)")
print(f"Saved: {SILVER_AIRPORTS} ({airports_silver.count()} rows)")

In [0]:
spark.table(SILVER_AIRLINES).show()
spark.table(SILVER_AIRPORTS).show(5)

## 7. Part 3 summary

Silver keeps cleaned flight data for both ML and BI:

- Cancelled and diverted flights are preserved;
- Missing departure delays are preserved;
- Duplicate records are removed.

Silver data tebles:

| Table | Name |
|-------|------|
| flights | `flight_delay.silver_flights` |
| airlines | `flight_delay.silver_airlines` |
| airports | `flight_delay.silver_airports` |

**Next steps:** 
- Gold - prepare data for BI dashboards.
- ML Preparation - filter flights and prepare features and target.